# Seminar 4. Overfitting, Validation, Custom NN Architectures

In this seminar we will look at:

1. **Validation**: How do we know our model is not just memorizing the training data (overfitting)? We will introduce a *Validation Set*.

2. **Custom Architectures**: Real-world data is complex. Often, we have multiple types of inputs (e.g., separate texts, numerical data). Instead of concatenating everything into one long string, we can build Neural Networks with multiple input branches.

3. **Advanced Blocks**: We will touch upon *Residual Connections* (ResNets), which help in training deeper networks.

## Environment Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import spacy
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
!python -m spacy download en_core_web_md -q

Dataset: https://www.kaggle.com/datasets/surendra365/recruitement-dataset

In [ ]:
df = pd.read_csv('/kaggle/input/recruitement-dataset/job_applicant_dataset.csv')
df

## EDA

In [ ]:
df.info()

In [ ]:
print('number of duplicates.')
for col in df.columns:
    print(f"{col}: {df[col].duplicated().sum()}")

In [ ]:
df = df.drop_duplicates(subset='Resume').reset_index(drop=True)
df.shape

In [ ]:
df['Best Match'].value_counts(normalize=True)

This time our data is pretty balanced again. In case of imbalance we would work with something from here: https://imbalanced-learn.org/stable/references/index.html

**Question**: How can we define what rebalancing technique should we use in any case? 

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Age distribution
axes[0].hist(df['Age'].dropna(), bins=20, edgecolor='black')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Text length analysis
df['Resume_Length'] = df['Resume'].apply(lambda x: len(str(x).split()))
df['JD_Length'] = df['Job Description'].apply(lambda x: len(str(x).split()))

axes[1].hist(df['Resume_Length'], bins=30, alpha=0.7, label='Resume', edgecolor='black')
axes[1].hist(df['JD_Length'], bins=30, alpha=0.7, label='Job Desc', edgecolor='black')
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

# Text statistics
print(f"Average resume length: {df['Resume_Length'].mean():.0f} words")
print(f"Average job description length: {df['JD_Length'].mean():.0f} words")

In [ ]:
df[['Resume', 'Job Description']].sample(15, random_state=42)

In [ ]:
df.sample(5, random_state=42)['Resume'].values

In [ ]:
df.sample(5, random_state=42)['Job Description'].values

**Question**: What else can we check?

## Preprocessing

In [ ]:
categorical_cols = ['Gender', 'Race', 'Ethnicity', 'Job Roles']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[f'{col}_encoded'] = le.fit_transform(df[col])
    label_encoders[col] = le

In [ ]:
nlp = spacy.load('en_core_web_md')
nlp.disable_pipes('ner', 'parser')
print(nlp.pipe_names)

In [ ]:
tqdm.pandas()

df['Resume'] = df['Resume'].progress_apply(lambda text: ' '.join([word.lemma_.lower() for word in nlp(text) if not word.is_stop and not word.is_punct]))
df['Job Description'] = df['Job Description'].progress_apply(lambda text: ' '.join([word.lemma_.lower() for word in nlp(text) if not word.is_stop and not word.is_punct]))

df.head()

In [ ]:
df['Resume'].duplicated().sum()

In [ ]:
df = df.drop_duplicates(subset='Resume').reset_index(drop=True)
df.shape

## Split Data

In order to monitor the training procedure of our neural network, we will need three sets:

1. Train (60%): To calculate gradients and update weights.

2. Validation (20%): To tune hyperparameters and stop training early (prevent overfitting).

3. Test (20%): To evaluate the final performance.

In [ ]:
# Features and Target
X = df.drop(columns='Best Match')
y = df['Best Match'].values

# Split: Train (60%) + Temp (40%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)

# Split Temp: Validation (20%) + Test (20%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train size: {X_train.shape[0]}")
print(f"Val size:   {X_val.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")

## Vectorization

We will vectorize resumes with Doc2Vec.

In [ ]:
from gensim.models.doc2vec import Doc2Vec,TaggedDocument

In [ ]:
tagged_resumes = [TaggedDocument(words=doc.split(), tags=[str(i)]) for i, doc in enumerate(X_train['Resume'])]
d2v_model = Doc2Vec(vector_size=50, min_count=2, epochs=20, seed=42)
d2v_model.build_vocab(tagged_resumes)
d2v_model.train(tagged_resumes, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

X_train['Resume Vector'] = [d2v_model.infer_vector(doc.split()) for doc in X_train['Resume']]
X_val['Resume Vector'] = [d2v_model.infer_vector(doc.split()) for doc in X_val['Resume']]
X_test['Resume Vector'] = [d2v_model.infer_vector(doc.split()) for doc in X_test['Resume']]

As for job descriptions, we will prepare them and leave for later. Right now we want to collect a bag of words from them.

In [ ]:
from collections import Counter

all_train_words = []
for text in X_train['Job Description']:
    all_train_words.extend(text.split())

vocab_count = Counter(all_train_words)

# Let's keep only words that appear at least twice to simulate real OOV scenarios
vocab = {word: i+2 for i, (word, count) in enumerate(vocab_count.most_common()) if count > 1}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

print(f"Vocabulary size: {len(vocab)}")

In [ ]:
# Helper function to apply the Train-Vocab to any text
MAX_SEQ_LEN = 20

def text_to_indices(text_series, vocab, max_len):
    matrix = []
    for text in text_series:
        tokens = text.split()
        # Crucial: If token is not in vocab, use <UNK> (1)
        indices = [vocab.get(token, vocab['<UNK>']) for token in tokens]
        
        # Padding/Truncating
        if len(indices) < max_len:
            indices = indices + [vocab['<PAD>']] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        matrix.append(indices)
    return matrix

In [ ]:
X_train['Job Description Indices'] = text_to_indices(X_train['Job Description'], vocab, MAX_SEQ_LEN)
X_val['Job Description Indices'] = text_to_indices(X_val['Job Description'], vocab, MAX_SEQ_LEN)
X_test['Job Description Indices'] = text_to_indices(X_test['Job Description'], vocab, MAX_SEQ_LEN)

print(f"Job Train indices: {X_train.shape}")
print(f"Example of Val indices (first row): {X_val['Job Description Indices'].values[0]}")

Also, let's take into account some additional data as it also can affect the employees' choices. As an example here, we will take age data into account, but we will have to normalize it first.

In [ ]:
# Normalize Age
scaler = StandardScaler()

# Fit on Train
X_train['Age Norm'] = scaler.fit_transform(X_train[['Age']])

# Transform Val and Test (using Train statistics)
X_val['Age Norm'] = scaler.transform(X_val[['Age']])
X_test['Age Norm'] = scaler.transform(X_test[['Age']])

In [ ]:
X_train.head()

## Simple Approach

### Defining Custom Datasets

To feed data into PyTorch, we need to wrap it in our custom dataset inherited from a `Dataset` class.

Custom dataset must have `__len__()` and `__getitem__()` methods defined.

In [ ]:
class JobDataset(Dataset):
    def __init__(self, resume_vecs, age_vecs, labels):
        self.resume_vecs = torch.tensor(resume_vecs, dtype=torch.float32)
        self.age_vecs = torch.tensor(age_vecs, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # Shape: (N, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'resume': self.resume_vecs[idx],
            'age': self.age_vecs[idx],
            'label': self.labels[idx]
        }

# Create datasets
train_dataset = JobDataset(np.array([list(vector) for vector in X_train['Resume Vector'].values]), [[val] for val in X_train['Age Norm'].values], y_train)
val_dataset = JobDataset(np.array([list(vector) for vector in X_val['Resume Vector'].values]), [[val] for val in X_val['Age Norm'].values], y_val)
test_dataset = JobDataset(np.array([list(vector) for vector in X_test['Resume Vector'].values]), [[val] for val in X_test['Age Norm'].values], y_test)

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### Baseline Model

In a simple approach, we will only use resume texts and age data. We will concatenate them into one vector and pass it through layers. This is how basic Feed-Forward Networks usually work.

In [ ]:
class SimpleBaseline(nn.Module):
    def __init__(self, input_dim):
        super(SimpleBaseline, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, resume, age):
        # Concatenate everything into one vector
        combined = torch.cat((resume, age), dim=1)
        return self.model(combined)

# Calculate total input dimension
input_dim_total = X_train['Resume Vector'].values[0].shape[0] + 1
simple_model = SimpleBaseline(input_dim_total).to(device)
print(simple_model)

Here we define the training function.
**Crucial Point**: We will track the loss on the **Validation Set**. If the validation loss starts increasing while training loss decreases, we are **overfitting**. We will save the model weights that gave the best validation loss.

In [ ]:
from tqdm.auto import trange

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=0.001):
    criterion = nn.BCELoss() # Binary Cross Entropy
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # History for plotting
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    best_val_loss = float('inf')
    best_model_state = None
    
    for epoch in trange(epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        for batch in tqdm(train_loader, desc='training'):
            resume = batch['resume'].to(device)
            age = batch['age'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(resume, age)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc='validating'):
                resume = batch['resume'].to(device)
                age = batch['age'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(resume, age)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                predicted = (outputs > 0.5).float()
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct / total
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_accuracy)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()
            
    print("Training complete. Restoring best weights...")
    model.load_state_dict(best_model_state)
    return history

# Train the baseline
print("Training Simple Baseline...")
history_simple = train_model(simple_model, train_loader, val_loader, epochs=15)

In [ ]:
def visualize_training_results(history, title="Model Training History"):
    """
    Plots the training and validation loss, and validation accuracy.
    Args:
        history (dict): Dictionary containing 'train_loss', 'val_loss', 'val_acc'.
        title (str): Title for the plots.
    """
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Create a figure with 2 subplots (1 row, 2 columns)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # --- Plot 1: Loss Curves ---
    ax1.plot(epochs, history['train_loss'], 'b-o', label='Training Loss')
    ax1.plot(epochs, history['val_loss'], 'r-o', label='Validation Loss')
    
    # Highlight the best epoch (min validation loss)
    best_epoch = np.argmin(history['val_loss']) + 1
    best_val_loss = np.min(history['val_loss'])
    
    ax1.axvline(x=best_epoch, color='green', linestyle='--', label=f'Best Epoch ({best_epoch})')
    ax1.scatter(best_epoch, best_val_loss, s=100, c='green', zorder=5)
    
    ax1.set_title(f'{title} - Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss (BCE)')
    ax1.legend()
    ax1.grid(True)
    
    # --- Plot 2: Validation Accuracy ---
    ax2.plot(epochs, history['val_acc'], 'g-s', label='Validation Accuracy')
    
    ax2.set_title(f'{title} - Validation Accuracy')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1.0) # Accuracy is between 0 and 1
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
visualize_training_results(history_simple, title="Simple Model")

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
simple_model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        resume = batch['resume'].to(device)
        age = batch['age'].to(device)
        labels = batch['label'].to(device)
        
        outputs = simple_model(resume, age)
        predicted = (outputs > 0.5).float()
        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        
print(classification_report(y_true, y_pred))

## Complex Approach

Now let's try something smarter.
Idea:

1. **Parallel Processing**: The Resume and the Job Description are processed by separate "branches" (streams). This allows the network to extract features from the resume *independently* before comparing it to the job.

2. **Multimodal Input**: Numerical data (Age) is added later in the pipeline.

3. **Residual Connection**: We add a skip connection (`x = x + layer(x)`). This is a foundational technique in modern Deep Learning (e.g., ResNet, Transformers) to prevent the gradient from vanishing and to allow the model to learn identity functions easier.

*(Note on Modern Architectures: In Multimodal LLMs, we often use separate encoders for images and text. In classic NLP, Siamese networks use this two-stream structure to measure similarity).*

### Define Hybrid Dataset

We will need another dataset structure for our idea to work

In [ ]:
class HybridJobDataset(Dataset):
    def __init__(self, resume_vecs, job_idxs, age_vecs, labels):
        # Doc2Vec vectors (Floats)
        self.resume_vecs = torch.tensor(resume_vecs, dtype=torch.float32)
        # Vocabulary Indices (Integers/Longs) for Embedding Layer
        self.job_idxs = torch.tensor(job_idxs, dtype=torch.long)
        # Numerical Age (Floats)
        self.age_vecs = torch.tensor(age_vecs, dtype=torch.float32)
        # Targets
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'resume': self.resume_vecs[idx],
            'job': self.job_idxs[idx],
            'age': self.age_vecs[idx],
            'label': self.labels[idx]
        }

# Create DataLoaders
train_ds = HybridJobDataset(
    np.array([list(vector) for vector in X_train['Resume Vector'].values]),
    np.array([list(idx) for idx in X_train['Job Description Indices'].values]), 
    [[val] for val in X_train['Age Norm'].values], 
    y_train
)
val_ds = HybridJobDataset(
    np.array([list(vector) for vector in X_val['Resume Vector'].values]), 
    np.array([list(idx) for idx in X_val['Job Description Indices'].values]), 
    [[val] for val in X_val['Age Norm'].values], 
    y_val
)
test_ds = HybridJobDataset(
    np.array([list(vector) for vector in X_test['Resume Vector'].values]), 
    np.array([list(idx) for idx in X_test['Job Description Indices'].values]), 
    [[val] for val in X_test['Age Norm'].values], 
    y_test
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

### Custom Neural Network with Embedding Layer

We will process our data in two different branches. Resume texts will be processed independently from job descriptions.

In [ ]:
class HybridArchitecture(nn.Module):
    def __init__(self, doc2vec_dim, vocab_size, embed_dim, age_dim):
        super(HybridArchitecture, self).__init__()
        
        # --- Branch 1: Resume (Pre-calculated Doc2Vec) ---
        # Input: [Batch, 50] -> Output: [Batch, 64]
        self.resume_branch = nn.Sequential(
            nn.Linear(doc2vec_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # --- Branch 2: Job Description (Learnable Embeddings) ---
        # 1. Embedding Layer: Converts indices to vectors.
        # Input: [Batch, Seq_Len] -> Output: [Batch, Seq_Len, Embed_Dim]
        self.job_embedding = nn.Embedding(num_embeddings=vocab_size, 
                                          embedding_dim=embed_dim, 
                                          padding_idx=0)
        
        # 2. Processing after averaging
        # Input: [Batch, Embed_Dim] -> Output: [Batch, 64]
        self.job_process = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU()
        )
        
        # --- Fusion & Residual Block ---
        combined_dim = 64 + 64 + age_dim
        
        self.fusion = nn.Linear(combined_dim, 64)
        
        self.res_block = nn.Sequential(
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 64)
        )
        
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, resume_vec, job_idx, age):
        # 1. Resume Stream
        r_emb = self.resume_branch(resume_vec)
        
        # 2. Job Stream
        # a. Get embeddings: [Batch, Seq_Len] -> [Batch, Seq_Len, Embed_Dim]
        j_emb = self.job_embedding(job_idx)
        
        # b. Aggregate: We need one vector per sentence. 
        # We take the MEAN across the sequence dimension (dim=1).
        # This works like "Bag of Embeddings".
        # [Batch, Seq_Len, Embed_Dim] -> [Batch, Embed_Dim]
        j_emb = torch.mean(j_emb, dim=1)
        
        # c. Process
        j_emb = self.job_process(j_emb)
        
        # 3. Concatenate (Resume + Job + Age)
        combined = torch.cat((r_emb, j_emb, age), dim=1)
        
        # 4. Fusion and Residual
        x = self.fusion(combined)
        identity = x
        out = self.res_block(x)
        x = x + out # Residual connection
        
        # 5. Output
        return self.classifier(x)

# Initialize Model
# doc2vec_dim = 50 (from our training)
# vocab_size = len(vocab)
# embed_dim = 32 (we choose this hyperparameter)
# age_dim = 1

model = HybridArchitecture(doc2vec_dim=50, 
                           vocab_size=len(vocab), 
                           embed_dim=32, 
                           age_dim=1).to(device)

print(model)

In [ ]:
print(f"\nNumber of parameters: {sum(p.numel() for p in model.parameters())}")

In [ ]:
print(f"\nNumber of parameters: {sum(p.numel() for p in simple_model.parameters())}")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=20, lr=0.001):
    criterion = nn.BCELoss() # Binary Cross Entropy
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # History for plotting
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    best_val_loss = float('inf')
    best_model_state = None
    
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            resume = batch['resume'].to(device)
            job = batch['job'].to(device)
            age = batch['age'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            outputs = model(resume, job, age)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_loader)
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                resume = batch['resume'].to(device)
                job = batch['job'].to(device)
                age = batch['age'].to(device)
                labels = batch['label'].to(device)
                
                outputs = model(resume, job, age)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                predicted = (outputs > 0.5).float()
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = correct / total
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_accuracy)
        
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()
            
    print("Training complete. Restoring best weights...")
    model.load_state_dict(best_model_state)
    return history

In [ ]:
# Train the Custom Model
print("\nTraining Custom Architecture...")
history_custom = train_model(model, train_loader, val_loader, epochs=15)

In [ ]:
visualize_training_results(history_custom, title="Custom Hybrid Model")

In [ ]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        resume = batch['resume'].to(device)
        job = batch['job'].to(device)
        age = batch['age'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(resume, job, age)
        predicted = (outputs > 0.5).float()
        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        
print(classification_report(y_true, y_pred))

Better example: https://colab.research.google.com/drive/1tlze5bMefGUGOU9-58RDW4shzHfWUL6N?usp=sharing